# Investigating the Properties of Static Word Embeddings through the Lens of Semantic Field Theory


In [1]:
import pandas as pd
import numpy as np
import umap

from sklearn.cluster import KMeans

import re
from gensim.models import KeyedVectors
from pymorphy3 import MorphAnalyzer
from russian_tagsets import converters

conv = converters.converter("opencorpora-int", "ud20")
morph = MorphAnalyzer()

from gensim.models import KeyedVectors

v2v_model = KeyedVectors.load_word2vec_format(
    "220/model.bin", binary=True
)  # ruwikiruscorpora_upos_cbow_300_10_2021

import pickle
import os
from sklearn.metrics import adjusted_rand_score
import plotly.express as px


## Создание датафрейма, предобработка и векторизация слов

In [21]:
def create_df(codes, filename="Shvedova.csv"):
    # поиск слов в базе данных по нужным кодам
    word_list = []
    code_list = []
    full_code = []
    Shvedova = pd.read_csv(filename)
    df = pd.DataFrame()
    for row in Shvedova.to_numpy():
        for code in codes:
            if row[2].startswith(code):
                word_list.append(row[1])
                code_list.append(code)
                full_code.append(row[2])
    df["word"] = word_list
    df["code"] = code_list
    df["full_code"] = full_code
    return df

In [22]:
codes = [
    "2.3.2.1",
    "2.3.2.2",
    "2.3.2.3",
]
df = create_df(codes)

In [23]:
df

,word,code,full_code
0,абстрагировать,2.3.2.1,2.3.2.1.2.1.
1,абстрагироваться,2.3.2.1,2.3.2.1.2.1.
2,алкать,2.3.2.3,2.3.2.3.3.
3,ассигновать,2.3.2.3,2.3.2.3.1.4.
4,ассоциировать,2.3.2.1,2.3.2.1.2.1.
...,...,...,...
1571,шалеть,2.3.2.2,2.3.2.2.4.2.2.
1572,шугать,2.3.2.3,2.3.2.3.1.2.1.3.
1573,щекотать,2.3.2.2,2.3.2.2.2.1.1.2.
1574,щемить,2.3.2.2,2.3.2.2.3.4.2.


In [24]:
# функция для препроцессинга и векторизации текстов
def preprocess_text(text):
    new_text = []
    for word in text:
        forms = morph.parse(word)[0]
        word = f"{word}_{conv(str(forms.tag)).split()[0]}"
        new_text.append(word)
    return " ".join(new_text)


def vectorize_text(text):
    text = preprocess_text(text).split()
    vecs = []
    no_words = []
    for word in text:
        if word in v2v_model:
            vecs.append(v2v_model[word])
        else:
            no_words.append(word)
    return np.vstack(vecs), no_words

In [25]:
X = vectorize_text(df["word"])[0]

In [26]:
# создаем дф, из слов, которые удалось векторизовать (векторы статические, поэтому не все слова есть в модели)
no_words = [el.split("_")[0] for el in vectorize_text(df["word"])[1]]
df_cleaned = df[~df["word"].isin(no_words)]

In [27]:
df_cleaned

,word,code,full_code
0,абстрагировать,2.3.2.1,2.3.2.1.2.1.
1,абстрагироваться,2.3.2.1,2.3.2.1.2.1.
2,алкать,2.3.2.3,2.3.2.3.3.
3,ассигновать,2.3.2.3,2.3.2.3.1.4.
4,ассоциировать,2.3.2.1,2.3.2.1.2.1.
...,...,...,...
1569,чуждаться,2.3.2.2,2.3.2.2.2.3.3.
1570,чуять,2.3.2.2,2.3.2.2.1.
1573,щекотать,2.3.2.2,2.3.2.2.2.1.1.2.
1574,щемить,2.3.2.2,2.3.2.2.3.4.2.


In [28]:
df_cleaned["code"].value_counts()

code
2.3.2.2    555
2.3.2.3    409
2.3.2.1    393
Name: count, dtype: int64

In [29]:
df_cleaned.to_csv("df_cleaned.csv", index=False)

In [30]:
len(set(df_cleaned["full_code"]))

77

In [ ]:
#визуализация выборки
df = df_cleaned.copy()

reducer = umap.UMAP(
    n_neighbors=15, min_dist=0.1, n_components=2, metric="cosine", random_state=42
)

embedding = reducer.fit_transform(X)
x = []
y = []
for el in embedding:
    x.append(el[0])
    y.append(el[1])

df['x']=x
df['y']=y



fig = px.scatter(
    df,
    y=df['y'],
    x=df['x'],
    color="code",
    title="Коды 2.3.2.1, 2.3.2.2, 2.3.2.3",
    hover_data={
        "word": True,
        "full_code": True,
    },
)



fig.show()

c:\Users\Napluch\OneDrive\Документы\cursach2\termpaper\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [31]:
df_cleaned

,word,code,full_code
0,абстрагировать,2.3.2.1,2.3.2.1.2.1.
1,абстрагироваться,2.3.2.1,2.3.2.1.2.1.
2,алкать,2.3.2.3,2.3.2.3.3.
3,ассигновать,2.3.2.3,2.3.2.3.1.4.
4,ассоциировать,2.3.2.1,2.3.2.1.2.1.
...,...,...,...
1569,чуждаться,2.3.2.2,2.3.2.2.2.3.3.
1570,чуять,2.3.2.2,2.3.2.2.1.
1573,щекотать,2.3.2.2,2.3.2.2.2.1.1.2.
1574,щемить,2.3.2.2,2.3.2.2.3.4.2.


## Кластеризация слов

In [32]:
def count_entropy(df, X, save_dir="models_dump_5repeats", n_repeats=5, base_seed=42):
    """
    получает на вход набор эмбеддингов для слов,
    1) сжимает пространство UMAP-ом до 50-мерного с шагом -50,
    2) кластеризует на каждое число кластеров от 3 до 89
    3) считает энтропию
    4) записывает результат в папку models_dump_5repeats: сохраняет метку кластера для каждого слова
    5) записывает результат в датафрейм: сохраняет среднюю полученную энтропию для каждого гиперпараметра
    6) для каждого гиперпараметра шаги 1-5 повторяются пять раз с разными фиксированными random state
    """

    os.makedirs(save_dir, exist_ok=True)
    run_id = 0
    result = []

    for repeat_id in range(n_repeats):
        umap_seed = base_seed + repeat_id
        kmeans_seed = umap_seed + 100

        for n_dimensions in range(300, 0, -50):
            for n_neighbors in [5, 10, 20, 100]:
                if n_dimensions == 300:
                    X_reduced = X
                    reducer_n = None
                else:
                    reducer_n = umap.UMAP(
                        n_neighbors=n_neighbors,
                        n_components=n_dimensions,
                        metric="cosine",
                        random_state=umap_seed,
                    )
                    X_reduced = reducer_n.fit_transform(X)
                for n_clusters in range(
                    3, 90
                ):  # ну больше кластеров мы точно не хотим, потому что у нас по Шведовой уникальных кодов 77. Но и это уже очень много
                    kmeans = KMeans(
                        n_clusters=n_clusters, n_init="auto", random_state=kmeans_seed
                    )
                    kmeans.fit(X_reduced)
                    labels = kmeans.labels_
                    df_local = df.copy()
                    df_local["cluster"] = labels
                    matrix = pd.crosstab(df_local["cluster"], df_local["code"])
                    probs = matrix.div(matrix.sum(axis=1), axis=0)
                    probs_without0 = probs.replace(0, 1)
                    ent = -(probs * np.log(probs_without0)).sum(axis=1)
                    mean_ent = ent.mean()

                    model_dump = {
                        "params": {
                            "n_dimensions": n_dimensions,
                            "n_neighbors": n_neighbors,
                            "n_clusters": n_clusters,
                        },
                        "seeds": {
                            "umap": umap_seed,
                            "kmeans": kmeans_seed,
                            "repeat_id": repeat_id,
                        },
                        "labels": labels,
                        "mean_entropy": mean_ent,
                    }

                    file_path = os.path.join(
                        save_dir,
                        f"run_{run_id}_{n_clusters}_{n_dimensions}_{n_neighbors}.pkl",
                    )
                    with open(file_path, "wb") as f:
                        pickle.dump(model_dump, f)

                    result.append(
                        {
                            "n_dimensions": n_dimensions,
                            "n_clusters": n_clusters,
                            "mean_ent": mean_ent,
                            "n_neighbors": n_neighbors,
                            "run_id": run_id,
                        }
                    )
                    run_id += 1

    return pd.DataFrame(result)

In [33]:
count_entropy(df_cleaned, X).to_csv("ent_5repeats_89clust.csv", index=False)

c:\Users\Napluch\OneDrive\Документы\cursach2\termpaper\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Napluch\OneDrive\Документы\cursach2\termpaper\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Napluch\OneDrive\Документы\cursach2\termpaper\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Napluch\OneDrive\Документы\cursach2\termpaper\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\Napluch\OneDrive\Документы\cursach2\termpaper\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed 

In [ ]:
df = pd.read_csv("ent_5repeats_90clust.csv")

fig = px.scatter(
    df,
    y="mean_ent",
    x="n_clusters",
    color="n_dimensions",
    title="Средняя энтропия по вектор-строке",
    hover_data={
        "n_clusters": True,
        "mean_ent": True,
        "n_dimensions": True,
        "n_neighbors": True,
        "run_id": True,
    },
)


fig.show()

## Анализ полученных результатов

In [2]:
def matr_code_label(
    run_id, path="models_dump_5repeats", df_cleaned=pd.read_csv("df_cleaned.csv")
):
    """
    по номеру дампа возвращает матрицу сопряженности код-кластер
    и датафрейм с меткой кластера для каждого слова
    """

    pattern = re.compile(rf"^run_{run_id}_\d+_\d+_\d+\.pkl$")

    for filename in os.listdir(path):
        if pattern.match(filename):
            file_path = os.path.join(path, filename)

    with open(file_path, "rb") as f:
        data = pickle.load(f)
    df_labels = df_cleaned.copy()
    df_labels["cluster"] = data["labels"]
    matrix = pd.crosstab(df_labels["cluster"], df_labels["code"])
    return matrix, df_labels

### Анализ конкретных точек на графике

In [4]:
# n_clusters = 7, mean_ent=0,57 (точка представляет собой выброс среди остальных значений энтропии)
df = matr_code_label(6699)[0]
df.T

cluster,0,1,2
code,,,
2.3.2.1,258,68,67
2.3.2.2,33,425,97
2.3.2.3,163,47,199


In [62]:
df = matr_code_label(2353)[1].to_csv('7clusters.csv')

In [8]:
# n_clusters = 36, mean_ent=0,53 (точка излома графика)
matr_code_label(6558)[0].T.to_csv('36clusters.csv')


In [ ]:
# n_clusters = 55, mean_ent=0,49 (локальный выброс)
df = matr_code_label(4489)[1]


0.05448884408297678